## 2.1 理论计算题

### 已知：
- 输入图像尺寸： 3 × 32 × 32 （通道数×高×宽）
- 卷积核：16个，每个大小  3 × 5 × 5（通道数×高×宽）
- 填充 \( p = 2 \)，步幅 \( s = 2 \)

### 1. 输出特征图尺寸

输出高度 $ H_{\text{out}}$ 和宽度 $ W_{\text{out}} $ 计算公式：

$$
H_{\text{out}} = \left\lfloor \frac{H_{\text{in}} + 2p - k}{s} \right\rfloor + 1
$$
$$
W_{\text{out}} = \left\lfloor \frac{W_{\text{in}} + 2p - k}{s} \right\rfloor + 1
$$

代入 $ H_{\text{out}}$ = 32 ，\( p = 2 \)，\( k = 5 \)，\( s = 2 \)：

$$
H_{\text{out}} = \left\lfloor \frac{32 + 4 - 5}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 15 + 1 = 16
$$

宽度同理得 16。输出通道数等于卷积核个数 16。

所以是： 16 × 16 × 16 

### 2. 单个输出像素的乘法次数

每个卷积核在输入上滑动，每个输出像素对应一个与卷积核相同大小的窗口。窗口内元素个数等于卷积核大小：

$$
\text{卷积核大小} = 3 \times 5 \times 5 = 75
$$

每个元素与卷积核对应权重相乘（点乘），共 75 次乘法。



In [19]:
#2.2 编程题
#手动实现支持 stride 和 padding 的二维最大池化前向传播（使用 NumPy）
import numpy as np

def max_pool2d(input, kernel_size, stride=None, padding=0):
    """
    二维最大池化前向传播（支持 batch, channel, height, width）
    参数:
        input: 4D numpy数组，形状 (N, C, H, W)
        kernel_size: int 或 (kh, kw)
        stride: int 或 (sh, sw)，默认为 kernel_size
        padding: int 或 (ph, pw)
    返回:
        output: 池化结果，形状 (N, C, H_out, W_out)
    """
    N, C, H, W = input.shape
    
    # 处理 kernel_size, stride, padding 为元组形式
    if isinstance(kernel_size, int):
        kh, kw = kernel_size, kernel_size
    else:
        kh, kw = kernel_size
    
    if stride is None:
        sh, sw = kh, kw
    elif isinstance(stride, int):
        sh, sw = stride, stride
    else:
        sh, sw = stride
    
    if isinstance(padding, int):
        ph, pw = padding, padding
    else:
        ph, pw = padding
    
    # 对输入进行零填充
    padded_H = H + 2 * ph
    padded_W = W + 2 * pw
    padded_input = np.pad(input, ((0, 0), (0, 0), (ph, ph), (pw, pw)), mode='constant')
    
    # 计算输出尺寸
    H_out = (padded_H - kh) // sh + 1
    W_out = (padded_W - kw) // sw + 1
    
    # 初始化输出
    output = np.zeros((N, C, H_out, W_out))
    
    # 滑动窗口取最大值
    for n in range(N):
        for c in range(C):
            for i in range(H_out):
                h_start = i * sh
                h_end = h_start + kh
                for j in range(W_out):
                    w_start = j * sw
                    w_end = w_start + kw
                    window = padded_input[n, c, h_start:h_end, w_start:w_end]
                    output[n, c, i, j] = np.max(window)
    return output

# 示例测试
if __name__ == "__main__":
    x = np.random.randn(2, 3, 32, 32)  # 模拟输入
    out = max_pool2d(x, kernel_size=3, stride=2, padding=1)
    print("输入形状:", x.shape)
    print("输出形状:", out.shape)  # 应为 (2, 3, 16, 16)

输入形状: (2, 3, 32, 32)
输出形状: (2, 3, 16, 16)


## 3.1 理论计算题
已知：输入输出通道数均为 $C$，无偏置。

### 1. 一个 $5 \times 5$ 卷积层参数量
卷积核大小 $5 \times 5$，输入通道 $C$，输出通道 $C$，参数量为：
$$
C \times C \times 5 \times 5 = 25C^2
$$

### 2. 两个串联 $3 \times 3$ 卷积层总参数量
每层：$C \times C \times 3 \times 3 = 9C^2$，两层共：
$$
2 \times 9C^2 = 18C^2
$$

In [20]:
#3.2 编程题
#使用 PyTorch 定义 NiN 块
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    NiN 块：一个普通卷积层 + 两个 1x1 卷积层，每层后跟 ReLU
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        kernel_size: 普通卷积的核大小
        stride: 普通卷积的步幅
        padding: 普通卷积的填充
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.block(x)

# 示例：输入通道3，输出通道16，3x3卷积，步幅1，填充1
block = NiNBlock(3, 16, 3, 1, 1)
x = torch.randn(1, 3, 32, 32)
out = block(x)
print(out.shape)  # torch.Size([1, 16, 32, 32])

torch.Size([1, 16, 32, 32])


## 4.1 理论计算题
已知：四个样本值 $x_1=2,x_2=4,x_3=6,x_4=8$，
$\gamma=2,\beta=1,\epsilon=0$。

### 计算过程
#### 均值
$$
\mu = \frac{2+4+6+8}{4}=5
$$

#### 方差（总体方差，$\epsilon=0$）
$$
\sigma^2=\frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}
=\frac{9+1+1+9}{4}=\frac{20}{4}=5
$$

#### 标准化
$$
\hat{x}_i=\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}=\frac{x_i-5}{\sqrt{5}}
$$

#### 输出
$$
y_i=\gamma \hat{x}_i+\beta=2\cdot\frac{x_i-5}{\sqrt{5}}+1
$$

#### 代入各值：
$$
\begin{align}
y_1 &= 2\cdot\frac{2-5}{\sqrt{5}}+1 = -\frac{6}{\sqrt{5}}+1 \\
y_2 &= 2\cdot\frac{4-5}{\sqrt{5}}+1 = -\frac{2}{\sqrt{5}}+1 \\
y_3 &= 2\cdot\frac{6-5}{\sqrt{5}}+1 = \frac{2}{\sqrt{5}}+1 \\
y_4 &= 2\cdot\frac{8-5}{\sqrt{5}}+1 = \frac{6}{\sqrt{5}}+1 \\
\end{align}
$$

### 所以是：
$y_1=1-\dfrac{6}{\sqrt{5}},\ y_2=1-\dfrac{2}{\sqrt{5}},\ y_3=1+\dfrac{2}{\sqrt{5}},\ y_4=1+\dfrac{6}{\sqrt{5}}$

In [24]:
#4.2 编程题
#PyTorch 实现残差块（Residual Block）
import torch
import torch.nn as nn
import torch.nn.functional as F

class Residual(nn.Module):          
    """
    残差块：两个 3x3 卷积层 + BN + ReLU，支持残差连接
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        stride: 第一个卷积的步幅（默认1）
        use_1x1conv: 是否使用 1x1 卷积调整残差支路
    """
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super(Residual, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        if use_1x1conv or in_channels != out_channels or stride != 1:
            self.shortcut = nn.Conv2d(in_channels, out_channels,
                                      kernel_size=1, stride=stride, bias=False)
        else:
            self.shortcut = None
    
    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        if self.shortcut is not None:
            identity = self.shortcut(x)
        
        out += identity
        out = F.relu(out)
        return out

# 示例测试（保持原样）
block = Residual(3, 16, stride=2, use_1x1conv=True)
x = torch.randn(1, 3, 32, 32)
out = block(x)
print(out.shape)  # torch.Size([1, 16, 16, 16])

torch.Size([1, 16, 16, 16])


### 5.1 理论计算题

**为什么底层学习率小、顶层学习率大？**

预训练模型的底层（靠近输入）学习通用特征（边缘、纹理等），对目标任务也有用，应小幅度更新或冻结，避免破坏已有知识。

顶层（输出层）随机初始化，需要快速拟合新任务的类别分布，因此设置较大学习率。

**目标数据集小且与源数据集相似时的微调策略**

- 冻结大部分底层特征提取层，只微调最后几层（或全连接层）。
- 使用更小的学习率，并增加数据增广（如翻转、裁剪等）以防止过拟合。
- 若数据集极小，可完全冻结所有预训练层，仅训练新初始化的输出层。

In [25]:
#5.2 编程题
#使用 torchvision.transforms 创建图像增广管道
from torchvision import transforms
from PIL import Image
import torch

# 定义增广组合
augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

# 打印输出示例：生成一张随机 PIL 图像并应用增广
fake_image = Image.fromarray((torch.rand(300, 300, 3) * 255).byte().numpy())
augmented_tensor = augmentation_pipeline(fake_image)
print("增广后张量形状:", augmented_tensor.shape)   # 应输出 torch.Size([3, 224, 224])

增广后张量形状: torch.Size([3, 224, 224])


## 6.1 理论计算题
已知：
真实框 $A = [10,10,50,50]$，预测框 $B = [30,30,70,70]$（格式：左上角 x, y，右下角 x, y）

### 1. 交集区域
左上角：$(\max(10,30),\max(10,30)) = (30,30)$
右下角：$(\min(50,70),\min(50,70)) = (50,50)$
交集宽高：$50 - 30 = 20$，面积 $20 \times 20 = 400$

### 2. 各自面积
$A$ 面积：$(50 - 10) \times (50 - 10) = 40 \times 40 = 1600$
$B$ 面积：$(70 - 30) \times (70 - 30) = 40 \times 40 = 1600$

### 3. 并集面积
$$1600 + 1600 - 400 = 2800$$

### 4. IoU
$$
\text{IoU} = \frac{400}{2800} = \frac{1}{7} \approx 0.142857
$$

In [23]:
#6.2 编程题
#实现标签平滑后的交叉熵损失函数
import torch
import torch.nn.functional as F

def label_smooth_cross_entropy(logits, labels, epsilon=0.1):
    """
    计算标签平滑后的交叉熵损失
    参数:
        logits: 模型输出，形状 (N, K)
        labels: 真实标签，形状 (N,)
        epsilon: 平滑因子，默认 0.1
    返回:
        标量损失值
    """
    N, K = logits.shape
    # 计算 log_softmax
    log_probs = F.log_softmax(logits, dim=1)
    
    # 构建平滑标签矩阵
    smooth_labels = torch.full_like(log_probs, epsilon / (K - 1))
    # 将正确类别位置的值设为 1 - epsilon
    smooth_labels.scatter_(1, labels.unsqueeze(1), 1 - epsilon)
    
    # 计算损失：- sum(smooth_labels * log_probs) 并求平均
    loss = - (smooth_labels * log_probs).sum(dim=1).mean()
    return loss

# 示例测试
logits = torch.randn(4, 10)   # 4个样本，10类
labels = torch.tensor([1, 3, 5, 7])
loss = label_smooth_cross_entropy(logits, labels, epsilon=0.1)
print("Label Smoothing CE Loss:", loss.item())

Label Smoothing CE Loss: 3.0133471488952637
